# **Data analysis on badges**

This notebook runs a bunch of data analysis on the badges and saves the analysis to `data/processed/badges`.

In [ ]:
import sys
import subprocess


subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "pandas",
    "plotly",
    "wordcloud",
    "matplotlib",
    "numpy",
    "kaleido",
    "pyarrow"
])

# Imports and setup

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import re
import numpy as np
import matplotlib.pyplot as plt  
from pathlib import Path
import json
import plotly.io as pio

df = pd.read_parquet('../data/user_badges.parquet')
index = {
    "numerical":{},
    "top":{},
    "graphs":{},
}

In [ ]:
users_df = pd.read_parquet('../data/users.parquet')

EXPORT_DIR = Path('../data/processed/badges').resolve()
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_WIDTH = 1100
DEFAULT_HEIGHT = 420

som_wide = go.layout.Template(layout=dict(
    width=DEFAULT_WIDTH,
    margin=dict(l=60, r=30, t=60, b=50),
))
pio.templates['som_wide'] = som_wide
pio.templates.default = 'plotly_white+som_wide'

try:
    pio.renderers.default = 'vscode'
except Exception:
    pass

def register_graph(key: str, name: str, description: str, image_file: str):
    index['graphs'][key] = {
        'name': name,
        'description': description,
        'image_file': Path(image_file).name
    }

def save_plotly(fig, name: str, description: str, scale: int = 2, width=None, height=None, friendly_name: str = ""):
    filename = f"{name}.png"
    out_path = EXPORT_DIR / filename
    try:
        exp_width = width if width is not None else (fig.layout.width or DEFAULT_WIDTH)
        exp_height = height if height is not None else (fig.layout.height or DEFAULT_HEIGHT)
        fig.write_image(str(out_path), scale=scale, width=exp_width, height=exp_height)
        register_graph(name, friendly_name, description, filename)
    except Exception as e:
        print(f'Failed to save Plotly figure {name}: {e}')

def save_matplotlib_current(name: str, description: str, dpi: int = 150, friendly_name: str = ""):
    filename = f"{name}.png"
    out_path = EXPORT_DIR / filename
    try:
        plt.gcf().savefig(out_path, bbox_inches='tight', dpi=dpi, facecolor='white')
        register_graph(name, friendly_name, description, filename)
    except Exception as e:
        print(f'Failed to save Matplotlib figure {name}: {e}')

def save_to_index(key: str, name: str, description: str, data, type="numerical"):
    if isinstance(data, pd.DataFrame):
        records = data.to_dict(orient='records')
        indexed = {i: rec for i, rec in enumerate(records)}
        index[type][key] = {
            'name': name,
            'description': description,
            'data': indexed,
        }
    else:
        index[type][key] = {
            'name': name,
            'description': description,
            'data': data
        }

print("Setup complete!")
print(f"Badge dataset shape: {df.shape}")
print(f"Users dataset shape: {users_df.shape}")

Setup complete!
Badge dataset shape: (2474, 5)
Users dataset shape: (21718, 16)


In [ ]:
df = df.merge(
    users_df[['id', 'display_name']], 
    left_on='user_id', 
    right_on='id', 
    how='left'
).drop('id', axis=1) 

df = df.rename(columns={'display_name': 'user_name'})

print("Added user names to badges dataframe")
print(f"Updated dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nSample with user names:")
df[['user_id', 'user_name', 'badge_name', 'badge_text']].head()

Added user names to badges dataframe
Updated dataset shape: (2474, 6)
Columns: ['user_id', 'badge_name', 'badge_text', 'badge_icon', 'scraped_at', 'user_name']

Sample with user names:


,user_id,user_name,badge_name,badge_text
0,1,Malted,<%= admin_tool do %>,with great power comes great responsibility......
1,1,Malted,Ballot Stuffer,vote 100 times.
2,1,Malted,Maiden Voyage,you shipped your first project! the journey be...
3,2,Rowan,<%= admin_tool do %>,with great power comes great responsibility......
4,2,Rowan,Gold Verified,this user is REALLY verified (i.e. gave us $32)


# **Numerical data**
Shows some numerical data

### Total badges count

Count the total number of badges in the dataset.

In [6]:
total_badges = len(df)
print(f"Total badges: {total_badges}")

save_to_index('total_badges', name='Total badges', description='The total number of badges in the dataset.', data=int(total_badges), type='numerical')

Total badges: 2474


### Unique badge types

Count the number of unique badge types.

In [7]:
unique_badges = df['badge_name'].nunique()
print(f"Unique badge types: {unique_badges}")

save_to_index('unique_badges', name='Unique badge types', description='The number of unique badge types in the dataset.', data=int(unique_badges), type='numerical')

Unique badge types: 16


### Users with badges

Count the number of unique users who have at least one badge.

In [8]:
users_with_badges = df['user_id'].nunique()
print(f"Users with badges: {users_with_badges}")

save_to_index('users_with_badges', name='Users with badges', description='The number of unique users who have at least one badge.', data=int(users_with_badges), type='numerical')

Users with badges: 2014


### Average badges per user

Calculate the average number of badges per user.

In [9]:
total_badges = len(df)
unique_users = df['user_id'].nunique()
avg_badges_per_user = total_badges / unique_users if unique_users > 0 else 0

print(f"Average badges per user: {avg_badges_per_user:.2f}")

save_to_index('avg_badges_per_user', name='Average badges per user', description='The average number of badges per user.', data=float(avg_badges_per_user), type='numerical')

Average badges per user: 1.23


# **Top stuff**

Shows the top stuff

### Top 3 users by badge count

Show the top 3 users with the most badges.

In [ ]:
_df = df.copy()
_df['user_name_clean'] = _df['user_name'].astype('object').fillna('(unnamed)').astype(str)

user_badge_counts = _df.groupby(['user_id', 'user_name_clean']).size().reset_index(name='badge_count')

top3 = (user_badge_counts.nlargest(3, 'badge_count').copy())
top3['profile_link'] = top3['user_id'].apply(
    lambda x: f"https://summer.hackclub.com/users/{x}")
top3.reset_index(drop=True, inplace=True)

print("Top 3 users by badge count:")

save_to_index('top_users_badges', name='Top 3 users by badge count',
              description='The top 3 users by number of badges.', data=top3, type='top')
top3

Top 3 users by badge count:


,user_id,user_name_clean,badge_count,profile_link
0,5,Neon,13,https://summer.hackclub.com/users/5
1,3,nora,9,https://summer.hackclub.com/users/3
2,14,Toshit,8,https://summer.hackclub.com/users/14


### Top 3 most common badges

Show the top 3 most frequently awarded badge types.

In [ ]:
badge_counts = df['badge_name'].value_counts().head(3).reset_index()
badge_counts.columns = ['badge_name', 'count']

badge_info = df.groupby('badge_name')['badge_text'].first().reset_index()
top3_badges = badge_counts.merge(badge_info, on='badge_name')

print("Top 3 most common badges:")

save_to_index('top_badges_common', name='Top 3 most common badges',
              description='The top 3 most frequently awarded badge types.', data=top3_badges, type='top')
top3_badges

Top 3 most common badges:


,badge_name,count,badge_text
0,Maiden Voyage,1995,you shipped your first project! the journey be...
1,Ballot Stuffer,245,vote 100 times.
2,Yapper I,59,Posted 10 comments on devlogs.


# **Graphs**

Various visualizations of badge data

### Badge distribution

A bar chart showing the distribution of badge types.

In [12]:
badge_counts = df['badge_name'].value_counts().reset_index()
badge_counts.columns = ['badge_name', 'count']

fig = px.bar(
    badge_counts.head(10),  # Top 10 badges
    x='badge_name',
    y='count',
    title='Top 10 Badge Types Distribution',
    labels={'badge_name': 'Badge Name', 'count': 'Number of Awards'}
)

fig.update_xaxes(tickangle=45)
fig.update_layout(height=500)
fig.show()

save_plotly(fig, 'badge_distribution', 'Distribution of the top 10 most common badge types', friendly_name='Badge distribution')

### Badges per user distribution

A histogram showing how many badges users typically have.

In [13]:
user_badge_counts = df.groupby('user_id').size().reset_index(name='badge_count')

fig = px.histogram(
    user_badge_counts,
    x='badge_count',
    nbins=30,
    title='Distribution of Badges per User',
    labels={'badge_count': 'Number of Badges', 'count': 'Number of Users'}
)

fig.update_layout(
    xaxis_title='Number of Badges per User',
    yaxis_title='Number of Users',
    height=500
)
fig.show()

save_plotly(fig, 'badges_per_user_distribution', 'Distribution showing how many badges users typically have', friendly_name='Badges per user distribution')

## Export

Save the analysis index to a JSON file.

### Badge statistics export

Export detailed statistics for each badge type including name, description, and award count.

In [ ]:
badge_stats = []

badge_counts = df['badge_name'].value_counts()

badge_descriptions = df.groupby('badge_name')['badge_text'].first()

for badge_name, count in badge_counts.items():
    badge_stat = {
        "badge_name": badge_name,
        "badge_description": badge_descriptions[badge_name],
        "num": int(count)
    }
    badge_stats.append(badge_stat)

badge_stats.sort(key=lambda x: x['num'], reverse=True)

index['badge_statistics'] = badge_stats

print(f"Badge statistics added to index")
print(f"Exported {len(badge_stats)} badge types")
print("\nTop 5 badges by award count:")
for i, badge in enumerate(badge_stats[:5]):
    print(f"{i+1}. {badge['badge_name']}: {badge['num']} awards")
    print(f"   Description: {badge['badge_description'][:100]}...")
    print()

Badge statistics added to index
Exported 16 badge types

Top 5 badges by award count:
1. Maiden Voyage: 1995 awards
   Description: you shipped your first project! the journey begins......

2. Ballot Stuffer: 245 awards
   Description: vote 100 times....

3. Yapper I: 59 awards
   Description: Posted 10 comments on devlogs....

4. Graphic Design is My Passion: 47 awards
   Description: Oh God How Did This Get Here I Am Not Good With Computer...

5. Spider: 39 awards
   Description: this user has a pet!...



In [ ]:
output_file = EXPORT_DIR / 'index.json'
with open(output_file, 'w') as f:
    json.dump(index, f, indent=2)

print(f"Analysis saved to {output_file}")

Analysis saved to /home/ajayanto/projects/SoM-Analytics/data/processed/badges/index.json
Generated 4 numerical insights
Generated 2 top lists
Generated 4 graphs
Exported 16 badge statistics
